# DistilBERT fine-tune — Food Hazard Detection (SemEval-2025 Task 9, ST1)

Neural baseline για το **Efialtis Stin Kouzina**. Φτιάχνω δύο ξεχωριστά
DistilBERT μοντέλα: ένα για το `hazard-category` και ένα για το
`product-category`, fine-tuned πάνω στο `title + text`.

Αντιστοιχεί στην ενότητα 12 του report. Το μοντέλο εκπαιδεύεται με σκέτο
cross-entropy, **χωρίς class weighting** (όπως στο report) — γι' αυτό και
υποαποδίδει στις σπάνιες κλάσεις που μετράει το macro-F1, και τελικά έχασε
από το TF-IDF + MiniLM stack.

## Οδηγίες (Colab)

1. **Runtime → Change runtime type → GPU** (T4 free αρκεί).
2. Στο Files panel ανέβασε τα `train.csv`, `valid.csv`, `test.csv` (από το
   `data/raw/`). Το notebook τα ψάχνει στο cwd ή σε `data/raw/`.
3. **Runtime → Run all**. Περίπου ~30-45 λεπτά σε T4.
4. Κατέβασε το `submission_distilbert.csv` και ανέβασέ το στο Kaggle.

Παράγει επίσης `logits_haz_*.npy`, `logits_prod_*.npy` και `label_maps.json`
για πιθανό ensemble αργότερα με το TF-IDF stack (ενότητα 14).

In [ ]:
!pip -q install "transformers>=4.40" "datasets>=2.19" accelerate

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# orizw configuration (idia logiki me thn enothta 12 tou report)
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256             # truncation - oi perissoteroi titloi+keimena xwrane aneta
BATCH = 16               # batch size sto fine-tune
EPOCHS = 4               # 4 epochs (sto report eipa oti isws 6-8 tha boithousan)
LR = 2e-5                # to klassiko learning rate gia BERT fine-tune
SEED = 42                # gia reproducibility

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("PROSOXH: xwris GPU to fine-tune einai poly argo. Runtime -> GPU.")

In [ ]:
# fortwnw ta dedomena. To train/valid exoun extra index column, to test oxi
# (idio me to src/io_utils.py tou topikou pipeline).
def vres_csv(name):
    for cand in (Path(name + ".csv"), Path("data/raw") / (name + ".csv")):
        if cand.exists():
            return cand
    raise FileNotFoundError(f"Anevase to {name}.csv (sto cwd i se data/raw/).")


def fortwse_split(name):
    if name in ("train", "valid"):
        return pd.read_csv(vres_csv(name), index_col=0)
    return pd.read_csv(vres_csv(name))


train = fortwse_split("train")
valid = fortwse_split("valid")
test = fortwse_split("test")
print(f"train={len(train)}  valid={len(valid)}  test={len(test)}")


def ftiakse_keimeno(df):
    # enwnw title + text. Den bazw metadata edw (to BERT to afhnw na dei
    # mono to keimeno, se antithesi me to TF-IDF pou pairnei kai ta metadata tokens).
    title = df.get("title", "").fillna("").astype(str)
    text = df.get("text", "").fillna("").astype(str)
    return (title + " " + text).str.strip()


for df in (train, valid, test):
    df["input_text"] = ftiakse_keimeno(df)

In [ ]:
# ftiaxnw ta label maps. Ta pairnw apo train+valid mazi wste na kalyfoun
# oles tis klaseis (kapoia rare class mporei na leipei apo to train mono).
pooled = pd.concat([train, valid], ignore_index=True)


def ftiakse_label_maps(col):
    klaseis = sorted(pooled[col].unique().tolist())
    label2id = {c: i for i, c in enumerate(klaseis)}
    id2label = {i: c for c, i in label2id.items()}
    return label2id, id2label


haz_l2i, haz_i2l = ftiakse_label_maps("hazard-category")
prod_l2i, prod_i2l = ftiakse_label_maps("product-category")
print(f"hazard classes={len(haz_l2i)}  product classes={len(prod_l2i)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)  # dynamic padding ana batch


# ftiaxnw ena aplo torch Dataset pou kanei tokenize ta keimena.
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = list(labels) if labels is not None else None

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item

In [ ]:
# orizw ta helpers gia training kai prediction.
def ekpaideuse_montelo(train_df, target_col, label2id, id2label, tag):
    # kanw fine-tune ena DistilBERT panw se ena apo ta dyo tasks.
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(label2id), id2label=id2label, label2id=label2id
    )
    ds = TextDataset(train_df["input_text"], train_df[target_col].map(label2id))
    args = TrainingArguments(
        output_dir=f"out_{tag}",
        per_device_train_batch_size=BATCH,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        fp16=torch.cuda.is_available(),   # fp16 sto GPU gia tahytita
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        seed=SEED,
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=ds,
        data_collator=collator, tokenizer=tokenizer,
    )
    trainer.train()
    return trainer


def provlepse_logits(trainer, texts):
    # gyrnaw ta raw logits (gia argmax kai gia possible ensemble argotera).
    ds = TextDataset(texts)
    return trainer.predict(ds).predictions


def metrhse_st1(y_haz_true, y_haz_pred, y_prod_true, y_prod_pred):
    # Episimo ST1: (macroF1(hazard) + macroF1(product | hazard swsto)) / 2
    f1_h = f1_score(y_haz_true, y_haz_pred, average="macro", zero_division=0)
    mask = np.asarray(y_haz_true) == np.asarray(y_haz_pred)
    if mask.sum() == 0:
        f1_p = 0.0
    else:
        f1_p = f1_score(np.asarray(y_prod_true)[mask], np.asarray(y_prod_pred)[mask],
                        average="macro", zero_division=0)
    return (f1_h + f1_p) / 2.0, f1_h, f1_p

In [ ]:
# kanw to (1): train sto train, axiologisi sto valid (single split).
haz_trainer = ekpaideuse_montelo(train, "hazard-category", haz_l2i, haz_i2l, "haz")
prod_trainer = ekpaideuse_montelo(train, "product-category", prod_l2i, prod_i2l, "prod")

haz_va_logits = provlepse_logits(haz_trainer, valid["input_text"])
prod_va_logits = provlepse_logits(prod_trainer, valid["input_text"])

haz_va_pred = np.array([haz_i2l[i] for i in haz_va_logits.argmax(1)])
prod_va_pred = np.array([prod_i2l[i] for i in prod_va_logits.argmax(1)])

st1, f1_h, f1_p = metrhse_st1(
    valid["hazard-category"].to_numpy(), haz_va_pred,
    valid["product-category"].to_numpy(), prod_va_pred,
)
print(f"\n[valid] ST1={st1:.4f}  hazard_F1={f1_h:.4f}  product_F1(cond)={f1_p:.4f}")
# Prosoxh: auto einai single-split, yperektimaei. Des enothta 13 (cross-validation).

In [ ]:
# kanw to (2): refit panw se train+valid kai prediction sto test gia to submission.
haz_full = ekpaideuse_montelo(pooled, "hazard-category", haz_l2i, haz_i2l, "haz_full")
prod_full = ekpaideuse_montelo(pooled, "product-category", prod_l2i, prod_i2l, "prod_full")

haz_te_logits = provlepse_logits(haz_full, test["input_text"])
prod_te_logits = provlepse_logits(prod_full, test["input_text"])

haz_te_pred = [haz_i2l[i] for i in haz_te_logits.argmax(1)]
prod_te_pred = [prod_i2l[i] for i in prod_te_logits.argmax(1)]

# grafw to Kaggle CSV me ta swsta columns.
submission = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "hazard-category": haz_te_pred,
    "product-category": prod_te_pred,
})
submission.to_csv("submission_distilbert.csv", index=False)
print("egrapsa submission_distilbert.csv", submission.shape)
submission.head()

In [ ]:
# sozw ta logits + ta label maps, gia na borw argotera na kanw soft-voting
# ensemble me to TF-IDF stack (enothta 14) xwris na ksanatrekso to BERT.
np.save("logits_haz_valid.npy", haz_va_logits)
np.save("logits_prod_valid.npy", prod_va_logits)
np.save("logits_haz_test.npy", haz_te_logits)
np.save("logits_prod_test.npy", prod_te_logits)

with open("label_maps.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "hazard": {str(i): c for i, c in haz_i2l.items()},
            "product": {str(i): c for i, c in prod_i2l.items()},
        },
        f, ensure_ascii=False, indent=2,
    )
print("eswsa logits_*.npy + label_maps.json")

## Σημειώσεις

- Το single-split valid ST1 εδώ υπερεκτιμά — δες την ενότητα 13 του report
  (cross-validation). Το τελικό Kaggle score του DistilBERT έπεσε κάτω από το
  TF-IDF + MiniLM stack.
- Για πιο δυνατό neural baseline: class weighting / focal loss στις σπάνιες
  κλάσεις, περισσότερα epochs με early stopping, και δοκιμή DeBERTa-v3.
- Τα `logits_*.npy` μπορούν να μπουν σε soft-voting ensemble με το
  `notebooks/12_stacking_ensemble.py`.